In [1]:
import os
# os.environ['CUDA_VISIBLE_DEVICES'] = "1"

import sys
print(os.getcwd())
sys.path.append('/root/data1/Code/Mumu/FlagEmbedding/FlagEmbedding/visual/VISTA_Evaluation_FineTuning-main/evaluation_example_webqa/BGE-base')
import faiss
import torch
import logging
import datasets
import numpy as np
from tqdm import tqdm
from typing import Optional
from dataclasses import dataclass, field
from transformers import HfArgumentParser
# from FlagEmbedding import FlagModel
from flag_eva_token_new import Flag_bgev_model
# from flag_clip import Flag_clip
import json

/root/data1/Code/Mumu/FlagEmbedding/FlagEmbedding/visual/VISTA_Evaluation_FineTuning-main/evaluation_example_webqa/BGE-base


/conda/envs/kgcopy/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/conda/envs/kgcopy/lib/python3.12/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [2]:
print(torch.cuda.is_available())

True


In [3]:
# parser = HfArgumentParser([Args])
# args: Args = parser.parse_args_into_dataclasses()[0]
resume_path = "/root/data1/Code/Mumu/save_rebuttal_model/new/checkpoint-5800/BGE_EVA_Token.pth"
# resume_path = "/root/data1/Code/Multimodalqa/alldata250108_save_models/checkpoint-500/BGE_EVA_Token.pth"
# resume_path = "/root/data1/Code/Mumu/save_models_caption/checkpoint-2900/BGE_EVA_Token.pth"
image_dir = "/root/data5/MuMuQA/resize_images263000/"
eval_data = datasets.load_dataset('json', data_files="/root/data5/MuMuQA/eval/test.json", split='train')
# mm_it_corpus = datasets.load_dataset('json',  data_files="the_path_to/mm_it_corpus.jsonl", split='train')
text_corpus = datasets.load_dataset('json', data_files="/root/data5/MuMuQA/eval/test.json", split='train')

model = Flag_bgev_model(model_name_bge = "BAAI/bge-base-en-v1.5",
                    model_name_eva = "EVA02-CLIP-B-16", # "EVA02-CLIP-B-16",
                    normlized = True,
                    eva_pretrained_path = "eva_clip",
                    resume_path=resume_path,
                    image_dir=image_dir,
                    )

/root/data1/Code/Mumu/FlagEmbedding/FlagEmbedding/visual/eva_clip/factory.py:87: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path, map_l

True
cuda


In [4]:
print(dir(model))  # 打印模型的所有属性和方法


['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', 'device', 'encode_corpus', 'encode_corpus_item', 'encode_image', 'encode_mm_it', 'encode_mm_it_mbeir', 'encode_queries', 'encode_text', 'image_dir', 'model', 'normalize_embeddings', 'num_gpus', 'pooling_method', 'tokenizer', 'use_fp16']


In [1]:
eval_data = datasets.load_dataset('json', data_files="/root/data1/Code/Mumu/processdata/retrive_goundtruth_ansnoNone.json", split='train')
    # mm_it_corpus = datasets.load_dataset('json',  data_files="the_path_to/mm_it_corpus.jsonl", split='train')
text_corpus = datasets.load_dataset('json', data_files="/root/data1/Code/Mumu/processdata/retrive_goundtruth_ansnoNone.json", split='train')

NameError: name 'datasets' is not defined

In [6]:
# print(eval_data["text"][0])
data = model.encode_corpus_item(eval_data["text"][0], batch_size = 1,max_length=512,corpus_type='text')
print(data.shape)

(14, 768)


In [8]:
extend = eval_data["image"][0].split('.')[-1]
img_id = eval_data["voa_image_id"][0] + '.'+extend
data = model.encode_queries([eval_data["question"][0],img_id], batch_size=1, max_length=512, query_type="mm_it")

In [9]:
print(data.shape)

(768,)


In [10]:
def search(model: Flag_bgev_model, queryQ,queryI, corpus,k:int = 100, batch_size: int = 1, max_length: int=512):
    """
    1. Encode queries into dense embeddings;
    2. Search through faiss index
    """
    # model.eval()
    with torch.no_grad():
        query_embedding = model.encode_queries([queryQ,queryI], batch_size=batch_size, max_length=max_length, query_type="mm_it")
        text_corpus = corpus
        
        text_corpus_embeddings = model.encode_corpus_item(text_corpus, batch_size=batch_size, max_length=max_length, corpus_type='text')
        # print(text_corpus_embeddings.shape)
        # print(query_embedding.shape)

        similarity = query_embedding @ text_corpus_embeddings.T
        # print(query_embedding.shape)
        # print(all_embeddings.shape)
        # print(similarity.shape)
        # print(type(similarity))
        # similarity = similarity.cpu().numpy()
        similarity = similarity.squeeze()
        top_indices = np.argsort(-similarity)[:k]
        # 获取对应的句子
        top_sentences = [text_corpus[i] for i in top_indices]

        return top_sentences

In [11]:
def append_to_json_file(data, file_path):
    # 检查文件是否存在
    if os.path.exists(file_path):
        # 如果文件存在，读取当前的JSON数据
        with open(file_path, "r") as f:
            try:
                file_data = json.load(f)  # 读取已有的JSON文件内容
            except json.JSONDecodeError:
                file_data = []  # 如果文件是空的或者格式错误，初始化为空列表
    else:
        file_data = []  # 文件不存在时，初始化为空列表

    # 将新数据追加到文件数据中
    file_data.append(data)

    # 将更新后的数据重新写入JSON文件
    with open(file_path, "w") as f:
        json.dump(file_data, f, indent=4)

In [9]:
from my_utils import *
ques_data =read_jsonl("/root/data5/MuMuQA/Entity_Description_FQuestion_deepseek.jsonl")
id2ques = dict()
for item in ques_data:
    data_id =item["data_id"]
    id2ques[data_id] = item["Rephrased Question"]
    # retrive_Sencond Question:
    # id2ques[data_id] = item["SecondQues"]
        # for idx,item in enumerate(tqdm(test_data[:])):
            # ques = id2ques[item["id"]]

In [13]:
print(eval_data[0])

{'data_id': '4791_2', 'question': 'What is the person in the black suit in the image warning people against?', 'caption': 'U.N. Secretary-General Antonio Guterres, right, shakes hands with Kenyan President Uhuru Kenyatta after holding a joint news conference at the State House in Nairobi, Kenya, March 8, 2017. \ufeff\ufeff\ufeffGuterres has called for more stable funding and support for African Union troops in Somalia, who are preventing extremists from taking over the country.', 'id': '4791_2', 'voa_image_id': 'VOA_EN_NW_2017_03_08_3755920_2', 'candidates': ['Guterres visited Somalia on Tuesday, where he said the situation was complicated. He said the combination of hunger, fatal diseases, drought, and the continuing struggle to defeat al-Shabab terrorists and to create conditions under which peace could be established "has had a devastating impact on the economy and in the lives of Somalis." People are suffering enormously, he said, and there is "a clear need of support from the inte

In [12]:
print(len(eval_data))
for idx,item in enumerate(tqdm(list(eval_data)[:])):
    # print(item)
    extend = item["image"].split('.')[-1]
    pic_id = item["voa_image_id"]+ '.'+extend
    # ques = "Caption : "+ item["caption"] +"Question : " + id2ques[item["data_id"]]
    ques = "Caption : "+ item["caption"] +"Question : " + item["question"]
    top_sentences = search(
            model=model, 
            queryQ=ques, 
            queryI = pic_id,
            corpus = item['text'],
            k=100,
            batch_size=1, 
            max_length=512
        )
    data = {
            'data_id': item["id"], 
            'question': ques, 
            'caption': item["caption"],
            'candidates': item["candidates"],
            'retrieved': top_sentences,
        }
    # print(data)
    # with open('/root/data1/Code/Mumu/retriveAnalysis/FirstQretrieve/retriveOq_vbge.jsonl',"a") as f:
    #     f.write(json.dumps(data)+"\n")
    append_to_json_file(data,'./train_rebuttal_mumu_new_step5800.json') 

1121


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1121/1121 [54:07<00:00,  2.90s/it]
